In [5]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import json

In [3]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
openai_keys = os.getenv("OPENAI_API_KEY")
if not openai_keys:
    raise ValueError("Please provide an OpenAI API key.")

In [4]:
## Model
model = ChatOpenAI(model="gpt-4o-mini")

In [6]:
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Path to your JSONL file
jsonl_file_path = "/Users/danielstephens/Desktop/Annotune-v2/annotune/synthetic-first-contact-plot-summaries-20241030-182950.jsonl"

def load_and_chunk_jsonl(jsonl_file_path):
    # Initialize the text splitter with desired chunk size
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=3000,  # Adjust chunk size based on your needs
        chunk_overlap=200  # Optional overlap between chunks
    )
    
    # Extract and chunk each document from the JSONL file
    chunks = []
    with open(jsonl_file_path, 'r') as file:
        for line in file:
            document = json.loads(line.strip())  # Load each JSON object
            
            # Extract fields
            summary = document.get("summary", "")
            setting = document.get("setting", "")
            style = document.get("style", "")
            themes = [document.get("theme_1", ""), document.get("theme_2", "")]
            mood=document.get("mood", "")
            
            if summary:
                # Split the summary text into chunks
                summary_chunks = text_splitter.split_text(summary)
                
                # Create structured chunks with additional metadata
                for chunk in summary_chunks:
                    structured_chunk = {

                        "setting": setting,
                        "style": style,
                        "themes": themes,
                        "mood" : mood,
                        "content": chunk
                    }
                    chunks.append(structured_chunk)
    
    return chunks

# Example usage
chunks = load_and_chunk_jsonl(jsonl_file_path)
for i, chunk in enumerate(chunks[:5]):  # Print the first few structured chunks as a sample
    print(f"Chunk {i+1}:")
    print(f"Setting: {chunk['setting']}")
    print(f"Style: {chunk['style']}")
    print(f"Themes: {chunk['themes']}")
    print(f"Mood: {chunk['mood']}")
    print(f"Content:\n{chunk['content']}\n")



Chunk 1:
Setting: Research facilities or laboratories: Controlled environments where scientists and experts can study and interact with the alien intelligence.
Style: Utopian/Dystopian: This style explores the implications of idealized or nightmarish societies, often serving as commentary on current social issues. Examples: George Orwell, Aldous Huxley, and Margaret Atwood.
Themes: ['Ethics and morality: Delving into the moral and ethical dilemmas that arise from encountering a non-human intelligence, such as the potential for exploitation or conflict.', "The impact on human identity: Investigating how contact with a non-human intelligence might challenge or change humanity's self-perception and sense of identity."]
Mood: intense
Content:
Caelum Research Institute, a state-of-the-art facility nestled in the heart of New Eden, has been at the forefront of intergalactic research since its inception in 2053. Dr. Sofia Jensen, a renowned astrobiologist, has dedicated her life to understand

In [ ]:
from langchain.vectorstores import Chroma
# Initialize embeddings and vector database
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_db = Chroma(collection_name="synthetic", persist_directory="database/chroma", embedding_function=embeddings)
for chunk in chunks:
    vector_db.add_texts([json.dumps(chunk)])


In [8]:
# Making a retriever 

retriever  = vector_db.as_retriever(search_kwargs={"k": 50})

In [20]:
import json

# Path to your JSONL file
jsonl_file_path = "/Users/danielstephens/Desktop/Annotune-v2/synthetic-first-contact-plot-summaries-20241023-130150.jsonl"

def get_unique_attributes(jsonl_file_path):
    # Sets to hold unique values
    unique_themes = set()
    unique_settings = set()
    unique_styles = set()
    
    # Read each JSON object line by line
    with open(jsonl_file_path, 'r') as file:
        for line in file:
            document = json.loads(line.strip())  # Load each JSON object
            
            # Add themes, setting, and style to respective sets
            theme_1 = document.get("theme_1", "")
            theme_2 = document.get("theme_2", "")
            setting = document.get("setting", "")
            style = document.get("style", "")
            
            if theme_1:
                unique_themes.add(theme_1)
            if theme_2:
                unique_themes.add(theme_2)
            if setting:
                unique_settings.add(setting)
            if style:
                unique_styles.add(style)
    
    # Convert sets to sorted lists for easier reading
    return {
        "themes": sorted(unique_themes),
        "settings": sorted(unique_settings),
        "styles": sorted(unique_styles)
    }

# Extract unique themes, settings, and styles
unique_attributes = get_unique_attributes(jsonl_file_path)
print("Unique Themes:", unique_attributes["themes"])
print("Unique Settings:", unique_attributes["settings"])
print("Unique Styles:", unique_attributes["styles"])


Unique Themes: ['Communication and understanding: Investigating the challenges and possibilities of communication between humans and non-human intelligences.', "Cultural and societal implications: Examining how humanity's institutions, values, and norms might be affected by contact with an alien intelligence.", 'Ethics and morality: Delving into the moral and ethical dilemmas that arise from encountering a non-human intelligence, such as the potential for exploitation or conflict.', "Humanity's place in the universe: Questioning humanity's significance, morality, and purpose in the face of a non-human intelligence.", 'The Other: Exploring the nature of the alien intelligence, its motivations, and its place in the universe.', "The impact on human identity: Investigating how contact with a non-human intelligence might challenge or change humanity's self-perception and sense of identity.", 'The unknown and the unknowable: Exploring the limits of human knowledge and understanding in the fa

In [220]:
#'Communication and understanding', "Cultural and societal implications", 'Ethics and morality', "Humanity's place in the universe", "The Other", "The impact on human identity", 
themes_ = ["The unknown and the unknowable"]

In [221]:
PROMPT_TEMPLATE = """
                    Each document in the context is tagged with specific themes,
                    and styles, which provide insight into its focus, location, and 
                    narrative approach. Based on the theme I will provide, 
                    generate 5 simple and general questions, with simple  answers 
                    that reflect meaningful inquiries across the data. The answers
                    should be succinct. 
                    Cite 10 different documents for each question that provide the answer.
                    Make sure the documents are quoted from the context
                    Return a json with the format below:

                        "question" : question,
                         "answer": answer,
                         "documents": 
                                'document1':
                                'document2':
                                'document3': ...
                                
                                
                                
                    Make sure the output is a correct json file. N
                    Extracted Context : {context}

                    
                    Theme: {theme}
                    """

In [222]:
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

In [223]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [224]:
model = ChatOpenAI(model="gpt-4o-mini")
chain = (
            {"context": retriever | format_docs, "theme": RunnablePassthrough()}
            | prompt
            | model
            | StrOutputParser()
         )

In [225]:
def get_questions(themes, chain):
    import json
    
    for theme_ in themes:
        number = 0
        while number < 10:

            response = chain.invoke("Theme: " + theme_)
            response = response[8:-3]
            
            output  = json.loads(response)
            print(output)
            try:
                with open("./savedFiles/"+theme_+str(number)+".json", "w") as f:
                    json.dump(output, f)
                number+=1
            except:
                break
            

In [227]:
get_questions(themes_, chain)

{'questions': [{'question': 'What challenges do humans face when trying to understand alien intelligences?', 'answer': 'Humans struggle with communication, perception differences, and the limits of their own knowledge.', 'documents': {'document1': 'In the end, Kymar and Dr. Quinlivan make a desperate bid to establish a new paradigm for human understanding, one that acknowledges the limits of human knowledge and the unknowable nature of The Devourer.', 'document2': "Dr. Soroka's team must navigate the consequences of their encounter with the Khronosphere, forever changed by their brush with the unknown.", 'document3': "The discovery of an extraterrestrial signal in the Atacama Desert's vast expanse sets off a chain reaction of events that will forever alter humanity's understanding of its place in the universe.", 'document4': "As tensions rise, Dr. Jain finds herself at the center of a brewing conflict between those who seek to exploit Khthon's power and those who advocate for caution."

#### Verifying the Questions Using LLM

In [42]:
def standardize_json(data):
    standardized = []
    
    # Check if data matches the first set format (list of dictionaries)
    if isinstance(data, list):
        for entry in data:
            standardized.append({
                "question": entry["question"],
                "answer": entry["answer"],
                "documents": entry["documents"]
            })
    
    # Check if data matches the second set format (dict with "questions" key)
    elif isinstance(data, dict) and "questions" in data:
        for entry in data["questions"]:
            standardized.append({
                "question": entry["question"],
                "answer": entry["answer"],
                "documents": entry["documents"]
            })
    
    return standardized




def replace_documents_with_full_summary(questions_data, full_data):
    for question in questions_data:
        for doc_key, doc_text in question["documents"].items():
            # Split document text by comma and clean each phrase
            phrases = [phrase.strip().replace(" ", "").replace(".", "") for phrase in doc_text.split(",")]
            match_found = False
            
            # Search through summaries in full_data
            for full_doc in full_data:
                summary_text = full_doc["summary"].replace(" ", "").replace(".", "")  # Clean summary text
                
                # Check sequences starting from each phrase
                for start_index in range(len(phrases)):
                    if all(phrase in summary_text for phrase in phrases[start_index:]):
                        question["documents"][doc_key] = full_doc["summary"]
                        match_found = True
                        break

                # If a match is found, stop checking other summaries
                if match_found:
                    break

            # If no sequence matches, combine phrases and split into two parts to check
            if not match_found:
                combined_text = "".join(phrases)  # Combine all phrases into a single cleaned string
                midpoint = len(combined_text) // 2
                part1 = combined_text[:midpoint].strip()
                part2 = combined_text[midpoint:].strip()

                # Search again in full_data for either half in the summary
                for full_doc in full_data:
                    summary_text = full_doc["summary"].replace(" ", "").replace(".", "")  # Clean summary text
                    if part1 in summary_text or part2 in summary_text:
                        question["documents"][doc_key] = full_doc["summary"]
                        break  # Replace and move to the next document if a match is found
    return questions_data










def read_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            # Parse each line as a JSON object
            data.append(json.loads(line))
    return data

In [31]:
full_data = read_jsonl("/Users/danielstephens/Desktop/Annotune-v2/annotune/synthetic-first-contact-plot-summaries-20241030-182950.jsonl")

In [26]:
import glob
files = glob.glob("/Users/danielstephens/Desktop/Annotune-v2/annotune/savedFiles/*")

In [43]:
for file in files:
    with open(file, "r") as f:
        data = json.load(f)
    standard_data = standardize_json(data)
    updated_questions_data = replace_documents_with_full_summary(standard_data, full_data)
    with open("/Users/danielstephens/Desktop/Annotune-v2/annotune/newSaved/"+file[61:], "w") as file:
        json.dump(updated_questions_data, file, indent=4)
    

In [14]:
file = "/Users/danielstephens/Desktop/Annotune-v2/annotune/savedFiles/Humanity's place in the universe4.json"